In [7]:
import pyscf
import pyqmc
import numpy as np
import deepqmc
import torch

In [8]:
# This setup is needed for the PySCF code
def setup_environment():
    import os
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1" 
    os.environ["OMP_NUM_THREADS"] = "1"
    
    for fname in ['mf.hdf5','optimized_wf.hdf5','vmc_data.hdf5','dmc.hdf5']:
        if os.path.isfile(fname): os.remove(fname)
    
    print("Environment configured: single-threaded execution")
    print("Previous calculation files cleaned up")
    
setup_environment()

Environment configured: single-threaded execution
Previous calculation files cleaned up


In [9]:
def create_h2_system():
    from pyscf import gto
    print("=" * 40 + "\nH2 Molecular System Setup\n" + "=" * 40)
    
    # Define H2 geometry, three dimensions
    mol = gto.M(
        atom=[['H', (0, 0, 0)], ['H', (0, 0, 1.4)]],  # This is specifying length of the bond between electrons
        basis='sto-3g'
    )
    
    print(f"H2 molecule created")
    print(f"  Atoms: {mol.natm}")
    print(f"  Electrons: {mol.nelectron}")
    print(f"  Bond length: 1.4 bohr (≈ {1.4 * 0.529:.2f} Angstrom)")
    print(f"  Basis set: STO-3G")
    print(f"  Nuclear repulsion: {mol.energy_nuc():.6f} Ha")
    
    return mol

h2_mol = create_h2_system()

H2 Molecular System Setup
H2 molecule created
  Atoms: 2
  Electrons: 2
  Bond length: 1.4 bohr (≈ 0.74 Angstrom)
  Basis set: STO-3G
  Nuclear repulsion: 0.377984 Ha


In [10]:
def Hartree_Fock_calculation(mol):
    from pyscf import scf
    print("=" * 40 + "\nH2 Mean-Field Calculation\n" + "=" * 40)
        
    # Run Hartree-Fock with pyscf build in function RHF
    mf = scf.RHF(mol)
    mf.chkfile = "mf.hdf5"
    energy = mf.kernel()
    
    print(f"HF calculation converged: E = {energy:.8f} Ha")
    print(f"  Nuclear repulsion: {mol.energy_nuc():.6f} Ha")
    print(f"  Electronic energy: {energy - mol.energy_nuc():.6f} Ha")
    print("Results saved to mf.hdf5")
    
    return mf

h2_mf = Hartree_Fock_calculation(h2_mol)

H2 Mean-Field Calculation
converged SCF energy = -0.941480654707799
HF calculation converged: E = -0.94148065 Ha
  Nuclear repulsion: 0.377984 Ha
  Electronic energy: -1.319464 Ha
Results saved to mf.hdf5


In [11]:
def wavefunction_optimization():
    """Optimization: Optimize Jastrow parameters using VMC"""
    import pyqmc.api as pyq
    print("=" * 40 + "\nWave Function Optimization\n" + "=" * 40)
    
    # Optimize with Slater-Jastrow wave function
    pyq.OPTIMIZE("mf.hdf5", "optimized_wf.hdf5", nconfig=100, max_iterations=10, verbose=True)
    
    # Read and display results
    df = pyq.read_opt("optimized_wf.hdf5")
    initial_e = df.iloc[0]['energy']
    final_e = df.iloc[-1]['energy'] 
    final_err = df.iloc[-1]['error']
    
    print(f"Optimization completed: {len(df)} iterations")
    print(f"  Initial energy: {initial_e:.6f} Ha")
    print(f"  Final energy: {final_e:.6f} +/- {final_err:.6f} Ha")
    print(f"  Energy lowering: {initial_e - final_e:.6f} Ha")
    return df

opt_df = wavefunction_optimization()

Wave Function Optimization
starting warmup
finished warmup
Was passed a single PGradAccumulator; using 1 sub_iteration. This is deprecated behavior.
#############################
Starting iteration 0 sub iteration 0
----------vmc done
Current energy -0.8896512149710928 0.012664721621879418
eigenvalues of Sij [2.52380616e-01 2.42296119e-01 1.41360879e-01 8.36817232e-02
 4.69236920e-02 1.44605070e-02 1.46286461e-02 1.22305645e-02
 1.27434939e-02 5.13243070e-03 1.59394499e-03 1.71866505e-03
 1.02265352e-03 1.17283477e-03 3.77758872e-04 9.56405358e-05
 1.10367378e-04 1.40031337e-04 1.46115961e-04 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00]
Gradient norm:  0.3520144665397139
Dot product between gradient and SR step:  0.9565704539703338
gradient [0.0405986  0.03685112 0.08054233 0.07276936 0.09442838 0.08496028
 0.05376275 0.04887794 0.02567086 0.02491389 0.05485233 0.05304624
 0.07173503 0.06705626 0.04275341 0.03865016 0.         0.15763966


In [12]:
def vmc_sampling():
    """VMC: Sample optimized wave function for accurate energy"""
    import pyqmc.api as pyq
    print("=" * 40 + "\nVMC Energy Sampling\n" + "=" * 40)
    
    # Run VMC with our optimized parameters
    pyq.VMC("mf.hdf5", "vmc_data.hdf5", load_parameters="optimized_wf.hdf5", 
            nblocks=30, verbose=False)
    
    # display results
    results = pyq.read_mc_output("vmc_data.hdf5")
    
    print(f"VMC completed successfully")
    print(f"  Total energy: {results['energytotal']:.8f} +/- {results['energytotal_err']:.8f} Ha")
    print(f"  Kinetic energy: {results['energyke']:.6f} +/- {results['energyke_err']:.6f} Ha")
    print(f"  Potential energy: {results['energytotal'] - results['energyke']:.6f} Ha")
    if 'acceptance' in results:
        print(f"  Acceptance rate: {results['acceptance']:.1%}")
    print(f"  Available data keys: {list(results.keys())}")
    return results

vmc_results = vmc_sampling()

VMC Energy Sampling
VMC completed successfully
  Total energy: -1.02550506 +/- 0.00145574 Ha
  Kinetic energy: 0.997220 +/- 0.004298 Ha
  Potential energy: -2.022725 Ha
  Acceptance rate: 83.7%
  Available data keys: ['fname', 'warmup', 'reblock', 'acceptance', 'acceptance_err', 'accumulator time', 'accumulator time_err', 'energyecp', 'energyecp_err', 'energyee', 'energyee_err', 'energyei', 'energyei_err', 'energygrad2', 'energygrad2_err', 'energyke', 'energyke_err', 'energytotal', 'energytotal_err', 'move time', 'move time_err']


In [13]:
def analyze_correlation_energy(vmc_results, hf_results, molecule_name="H2"):
    """Analysis: Calculate and display correlation energy recovery"""
    print("=" * 40 + f"\n{molecule_name} Correlation Analysis\n" + "=" * 40)
    
    # Extract energies
    hf_energy = hf_results.e_tot
    vmc_energy = vmc_results['energytotal']
    vmc_error = vmc_results['energytotal_err']
    
    # Calculate correlation energy (energy lowering from HF to VMC)
    correlation_energy = vmc_energy - hf_energy
    
    print(f"Hartree-Fock energy: {hf_energy:.8f} Ha")
    print(f"VMC energy: {vmc_energy:.8f} +/- {vmc_error:.8f} Ha")
    print(f"Correlation energy: {correlation_energy:.6f} Ha")
    
    # Convert to more intuitive units
    correlation_kcal = correlation_energy * 627.509  # Ha to kcal/mol
    print(f"Correlation energy: {correlation_kcal:.2f} kcal/mol")
    
    # Analysis of the result
    if abs(correlation_energy) > 0.01:  # Significant correlation
        print(f"VMC successfully recovered {abs(correlation_energy):.6f} Ha of correlation energy")
        print("This represents the energy gained by allowing electrons to correlate their motion (interact with each other)")
    else:
        print("Minimal correlation energy recovered - system may be well-described by HF")
    
    return correlation_energy

h2_corr_energy = analyze_correlation_energy(vmc_results, h2_mf, "H2")


H2 Correlation Analysis
Hartree-Fock energy: -0.94148065 Ha
VMC energy: -1.02550506 +/- 0.00145574 Ha
Correlation energy: -0.084024 Ha
Correlation energy: -52.73 kcal/mol
VMC successfully recovered 0.084024 Ha of correlation energy
This represents the energy gained by allowing electrons to correlate their motion (interact with each other)


In [14]:
def create_lih_system():
    from pyscf import gto
    print("=" * 40 + "\nLiH Molecular System Setup\n" + "=" * 40)
    
    atoms = [['Li', (0, 0, 0)], ['H', (0, 0, 3.0)]]
    
    mol = gto.M(atom=atoms, basis='sto-3g')
    
    print(f"LiH molecule created")
    print(f"  Atoms: {mol.natm} ({mol.atom_symbol(0)}, {mol.atom_symbol(1)})")
    print(f"  Electrons: {mol.nelectron}")
    print(f"  Bond length: 3.0 bohr (≈ {3.0 * 0.529:.2f} Å)")
    print(f"  Basis set: STO-3G")
    print(f"  Nuclear repulsion: {mol.energy_nuc():.6f} Ha")
    print("  Electronic structure: Li 1s²2s¹ + H 1s¹ → Li⁺H⁻-like bonding")
    
    return mol

def lih_Hartree_Fock_calculation(mol):
    from pyscf import scf
    print("=" * 40 + "\nLiH Mean-Field Calculation\n" + "=" * 40)
    
    print("[STARTING] LiH Hartree-Fock calculation...")
    print("  Challenge: 4 electrons, heteronuclear bonding")
    
    mf = scf.RHF(mol)
    mf.chkfile = 'lih_mf.hdf5'
    energy = mf.kernel()
    
    print(f"HF calculation converged: E = {energy:.8f} Ha")
    print(f"  Nuclear repulsion: {mol.energy_nuc():.6f} Ha")
    print(f"  Electronic energy: {energy - mol.energy_nuc():.6f} Ha")
    print(f"Results saved to lih_mf.hdf5")
    
    return mf

def lih_optimization():
    import pyqmc.api as pyq
    print("=" * 40 + "\nLiH Wave Function Optimization\n" + "=" * 40)
    
    pyq.OPTIMIZE("lih_mf.hdf5", "lih_optimized_wf.hdf5", nconfig=100, max_iterations=10, verbose=False)
    
    df = pyq.read_opt("lih_optimized_wf.hdf5")
    initial_e = df.iloc[0]['energy']
    final_e = df.iloc[-1]['energy'] 
    final_err = df.iloc[-1]['error']
    
    print(f"LiH optimization completed: {len(df)} iterations")
    print(f"  Initial energy: {initial_e:.6f} Ha")
    print(f"  Final energy: {final_e:.6f} +/- {final_err:.6f} Ha")
    print(f"  Energy lowering: {initial_e - final_e:.6f} Ha")
    return df

def lih_vmc_sampling():
    import pyqmc.api as pyq
    print("=" * 40 + "\nLiH VMC Energy Sampling\n" + "=" * 40)
    
    pyq.VMC("lih_mf.hdf5", "lih_vmc_data.hdf5", load_parameters="lih_optimized_wf.hdf5", 
            nblocks=30, verbose=False)
    
    results = pyq.read_mc_output("lih_vmc_data.hdf5")
    
    print(f"LiH VMC completed successfully")
    print(f"  Total energy: {results['energytotal']:.8f} +/- {results['energytotal_err']:.8f} Ha")
    print(f"  Kinetic energy: {results['energyke']:.6f} +/- {results['energyke_err']:.6f} Ha")
    print(f"  Potential energy: {results['energytotal'] - results['energyke']:.6f} Ha")
    if 'acceptance' in results:
        print(f"  Acceptance rate: {results['acceptance']:.1%}")
    return results


def comparison_summary(h2_results, lih_results, h2_mf, lih_mf):
    print("=" * 50 + "\nH2 vs LiH Comparison Summary\n" + "=" * 50)
    
    print("Hartree-Fock Energies:")
    print(f"  H2:  {h2_mf.e_tot:.8f} Ha")
    print(f"  LiH: {lih_mf.e_tot:.8f} Ha")
    
    print(f"\nVMC Energies:")
    print(f"  H2:  {h2_results['energytotal']:.8f} +/- {h2_results['energytotal_err']:.8f} Ha")
    print(f"  LiH: {lih_results['energytotal']:.8f} +/- {lih_results['energytotal_err']:.8f} Ha")
    
    h2_corr = h2_results['energytotal'] - h2_mf.e_tot
    lih_corr = lih_results['energytotal'] - lih_mf.e_tot
    print(f"\nElectron Correlation Energy (VMC - HF):")
    print(f"  H2:  {h2_corr:.6f} Ha")
    print(f"  LiH: {lih_corr:.6f} Ha")
    
    print("=" * 50 + "\nComparison complete!\n" + "=" * 50)

lih_mol = create_lih_system()
lih_mf = lih_Hartree_Fock_calculation(lih_mol)
lih_opt_df = lih_optimization()
lih_vmc_results = lih_vmc_sampling()
lih_corr_energy = analyze_correlation_energy(lih_vmc_results, lih_mf, "LiH")

LiH Molecular System Setup
LiH molecule created
  Atoms: 2 (Li, H)
  Electrons: 4
  Bond length: 3.0 bohr (≈ 1.59 Å)
  Basis set: STO-3G
  Nuclear repulsion: 0.529177 Ha
  Electronic structure: Li 1s²2s¹ + H 1s¹ → Li⁺H⁻-like bonding
LiH Mean-Field Calculation
[STARTING] LiH Hartree-Fock calculation...
  Challenge: 4 electrons, heteronuclear bonding
converged SCF energy = -7.71082990021723
HF calculation converged: E = -7.71082990 Ha
  Nuclear repulsion: 0.529177 Ha
  Electronic energy: -8.240007 Ha
Results saved to lih_mf.hdf5
LiH Wave Function Optimization
LiH optimization completed: 10 iterations
  Initial energy: -7.512157 Ha
  Final energy: -7.930196 +/- 0.031003 Ha
  Energy lowering: 0.418039 Ha
LiH VMC Energy Sampling
LiH VMC completed successfully
  Total energy: -7.90864026 +/- 0.00870724 Ha
  Kinetic energy: 8.005439 +/- 0.116662 Ha
  Potential energy: -15.914079 Ha
  Acceptance rate: 58.6%
LiH Correlation Analysis
Hartree-Fock energy: -7.71082990 Ha
VMC energy: -7.90864026 +/

In [15]:
def comparison_summary(h2_results, lih_results, h2_mf, lih_mf):
    """Summary: Compare H2 vs LiH VMC results"""
    print("=" * 50 + "\nH2 vs LiH Comparison Summary\n" + "=" * 50)
    
    # HF energies comparison
    print("Hartree-Fock Energies:")
    print(f"  H2:  {h2_mf.e_tot:.8f} Ha")
    print(f"  LiH: {lih_mf.e_tot:.8f} Ha")
    
    # VMC energies comparison
    print(f"\nVMC Energies:")
    print(f"  H2:  {h2_results['energytotal']:.8f} +/- {h2_results['energytotal_err']:.8f} Ha")
    print(f"  LiH: {lih_results['energytotal']:.8f} +/- {lih_results['energytotal_err']:.8f} Ha")
    
    # Correlation energy (VMC - HF)
    h2_corr = h2_results['energytotal'] - h2_mf.e_tot
    lih_corr = lih_results['energytotal'] - lih_mf.e_tot
    print(f"\nElectron Correlation Energy (VMC - HF):")
    print(f"  H2:  {h2_corr:.6f} Ha")
    print(f"  LiH: {lih_corr:.6f} Ha")
    
    print("=" * 50)

comparison_summary(vmc_results, lih_vmc_results, h2_mf, lih_mf)

H2 vs LiH Comparison Summary
Hartree-Fock Energies:
  H2:  -0.94148065 Ha
  LiH: -7.71082990 Ha

VMC Energies:
  H2:  -1.02550506 +/- 0.00145574 Ha
  LiH: -7.90864026 +/- 0.00870724 Ha

Electron Correlation Energy (VMC - HF):
  H2:  -0.084024 Ha
  LiH: -0.197810 Ha


In [16]:
def test_ml_imports():
    """Test: Verify all ML-VMC dependencies are working"""
    print("=" * 50 + "\nTesting ML-VMC Imports\n" + "=" * 50)
    
    try: import jax; print("JAX version:", jax.__version__)
    except ImportError as e: print("ERROR: JAX import failed:", e)
    try: import haiku; print("Haiku version:", haiku.__version__)
    except ImportError as e: print("ERROR: Haiku import failed:", e)
    try: import deepqmc; print("DeepQMC imported successfully")
    except ImportError as e: print("ERROR: DeepQMC import failed:", e)
    try: import hydra; print("Hydra version:", hydra.__version__)
    except ImportError as e: print("ERROR: Hydra import failed:", e)
    
    try:
        import jax
        devices = jax.devices()
        device_info = f"{len(devices)} device(s): {[d.device_kind for d in devices]}"
        print(f"JAX devices: {device_info}")
        if any(d.device_kind == 'gpu' for d in devices):
            print("[ACCELERATED] GPU acceleration available!")
        else:
            print("[INFO] Running on CPU (GPU acceleration requires CUDA-enabled JAX)")
    except Exception as e: print("[WARNING] JAX device info failed:", e)
    
    print("=" * 50 + "\nML import testing complete!\n" + "=" * 50)

test_ml_imports()

Testing ML-VMC Imports
JAX version: 0.5.3
Haiku version: 0.0.14
DeepQMC imported successfully
Hydra version: 1.3.2
JAX devices: 1 device(s): ['cpu']
[INFO] Running on CPU (GPU acceleration requires CUDA-enabled JAX)
ML import testing complete!


In [17]:
def setup_ml_environment():
    """Setup: Configure JAX and clean up old ML runs"""
    import os
    import shutil
    
    # JAX configuration for reproducibility
    os.environ["JAX_ENABLE_X64"] = "True"  # Double precision
    
    # Clean up any existing ML runs
    for dirname in ['h2_ml_runs', 'lih_ml_runs', 'runs']:
        if os.path.exists(dirname): shutil.rmtree(dirname)
    
    print("JAX configured: double precision enabled")
    print("Previous ML calculation directories cleaned up")

setup_ml_environment()

JAX configured: double precision enabled
Previous ML calculation directories cleaned up


In [18]:
def create_h2_molecule():
    from deepqmc.molecule import Molecule
    from deepqmc.hamil import MolecularHamiltonian
    print("=" * 40 + "\nH2 Molecule Setup (ML-VMC)\n" + "=" * 40)
    
    # Create H2 molecule (same geometry as traditional VMC)
    mol = Molecule(
        coords=[[0.0, 0.0, 0.0], [0.0, 0.0, 0.74]], 
        charges=[1, 1], 
        charge=0, 
        spin=0, 
        unit='angstrom'
    )
    
    print(f"H2 molecule created: bond length = 0.74 A")
    print(f"  Coordinates: {mol.coords}")
    print(f"  Charges: {mol.charges}, Total charge: {mol.charge}, Spin: {mol.spin}")
    
    # Create Hamiltonian
    H = MolecularHamiltonian(mol=mol)
    print("Molecular Hamiltonian created")
    print("Ready for neural wave function setup")
    
    return mol, H

mol_h2, H_h2 = create_h2_molecule()

H2 Molecule Setup (ML-VMC)
H2 molecule created: bond length = 0.74 A
  Coordinates: [[0.        0.        0.       ]
 [0.        0.        1.3983973]]
  Charges: [1. 1.], Total charge: 0, Spin: 0
Molecular Hamiltonian created
Ready for neural wave function setup


In [19]:
def setup_neural_ansatz(H):
    import os
    import haiku as hk
    from hydra import compose, initialize_config_dir
    from hydra.utils import instantiate
    from deepqmc.app import instantiate_ansatz
    import deepqmc
    
    print("=" * 40 + "\nNeural Wave Function Setup\n" + "=" * 40)
    
    # Load default neural ansatz configuration
    deepqmc_dir = os.path.dirname(deepqmc.__file__)
    config_dir = os.path.join(deepqmc_dir, 'conf/ansatz')
    
    with initialize_config_dir(version_base=None, config_dir=config_dir):
        cfg = compose(config_name='default')
    
    print("Loaded default ansatz configuration")
    print(f"  Config type: {cfg._target_}")
    
    # Instantiate the neural ansatz
    _ansatz = instantiate(cfg, _recursive_=True, _convert_='all')
    ansatz = instantiate_ansatz(H, _ansatz)
    
    print("Neural wave function ansatz created")
    print("  Architecture: Graph Neural Network with electron embeddings")
    print("  Type: Slater-Jastrow-like with learned orbitals")
    
    return ansatz

ansatz_h2 = setup_neural_ansatz(H_h2)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Neural Wave Function Setup
Loaded default ansatz configuration
  Config type: deepqmc.wf.NeuralNetworkWaveFunction
Neural wave function ansatz created
  Architecture: Graph Neural Network with electron embeddings
  Type: Slater-Jastrow-like with learned orbitals


In [20]:
def setup_ml_sampler():
    from deepqmc.sampling import initialize_sampling, combine_samplers, MetropolisSampler, DecorrSampler
    from functools import partial
    
    print("=" * 40 + "\nML-Enhanced Sampler Setup\n" + "=" * 40)
    
    # Create combined sampler (Metropolis + decorrelation) 
    elec_sampler = partial(combine_samplers, samplers=[
        DecorrSampler(length=10),
        MetropolisSampler
    ])
    sampler_factory = partial(initialize_sampling, elec_sampler=elec_sampler)
    
    print("Advanced sampler factory created")
    print("  Components: DecorrSampler(10) + MetropolisSampler") 
    print("  Features: Automatic decorrelation + adaptive step size")
    
    return sampler_factory

sampler_factory = setup_ml_sampler()

ML-Enhanced Sampler Setup
Advanced sampler factory created
  Components: DecorrSampler(10) + MetropolisSampler
  Features: Automatic decorrelation + adaptive step size


In [21]:
def setup_optimizer():
    import os
    from hydra import compose, initialize_config_dir
    from hydra.utils import instantiate
    import deepqmc
    
    print("=" * 40 + "\nKFAC Optimizer Setup\n" + "=" * 40)
    
    # Load KFAC optimizer configuration
    deepqmc_dir = os.path.dirname(deepqmc.__file__)
    opt_config_dir = os.path.join(deepqmc_dir, 'conf/task/opt')
    
    with initialize_config_dir(version_base=None, config_dir=opt_config_dir):
        opt_cfg = compose(config_name='kfac')
    
    optimizer = instantiate(opt_cfg, _recursive_=True, _convert_='all')
    
    print("KFAC optimizer loaded")
    print("  Type: Kronecker-Factored Approximate Curvature")
    print("  Features: Second-order optimization for neural networks")
    
    return optimizer

optimizer = setup_optimizer()

KFAC Optimizer Setup
KFAC optimizer loaded
  Type: Kronecker-Factored Approximate Curvature
  Features: Second-order optimization for neural networks


In [24]:
#### Train the neural wavefunction on H2

def train_h2_neural_wf(H, ansatz, optimizer, sampler_factory):
    from deepqmc.train import train
    print("=" * 40 + "\nH2 Neural Wave Function Training\n" + "=" * 40)
    
    print("Neural wave function training...")
    print("  System: H2 molecule")
    print("  Method: Variational Monte Carlo with neural ansatz")
    
    # Run training with progress monitoring
    try:
        train(
            H, ansatz, optimizer, sampler_factory,
            steps=500,                   
            electron_batch_size=500,     
            seed=42,                      
            workdir='h2_ml_runs',
            pretrain_steps=100,
            max_restarts=2
        )
        print("Training completed successfully!")
        
    except Exception as e:
        print(f"Training encountered issue: {e}")
        print("  This is normal for first-time ML-VMC runs")
    
    print("Results saved to h2_ml_runs/ directory")

train_h2_neural_wf(H_h2, ansatz_h2, optimizer, sampler_factory)

H2 Neural Wave Function Training
Neural wave function training...
  System: H2 molecule
  Method: Variational Monte Carlo with neural ansatz
Training encountered issue: LayerTag.__init__() takes 1 positional argument but 4 were given
  This is normal for first-time ML-VMC runs
Results saved to h2_ml_runs/ directory


In [23]:
def analyze_h2_results():
    import h5py
    import numpy as np
    import os
    import matplotlib.pyplot as plt
    
    print("=" * 40 + "\nH2 ML-VMC Results Analysis\n" + "=" * 40)
    
    results_file = 'h2_ml_runs/training/result.h5'
    
    if not os.path.exists(results_file):
        print("Results file not found - training may not have completed")
        return None
    
    with h5py.File(results_file, 'r') as f:
        print("Results file found, analyzing data...")
        print(f"  Available data: {list(f.keys())}")
        
        if 'local_energy' in f:
            energy_data = f['local_energy']
            print(f"  Energy data keys: {list(energy_data.keys())}")
            
            if 'mean' in energy_data:
                energy_means = np.array(energy_data['mean']).flatten()
                energy_stds = np.array(energy_data['std']).flatten() if 'std' in energy_data else np.zeros_like(energy_means)
                steps = np.arange(len(energy_means))
                
                final_energy = energy_means[-1]
                energy_std = energy_stds[-1]
                
                print(f"Final ML-VMC energy: {final_energy:.8f} +/- {energy_std:.8f} Ha")
                print(f"  Training convergence: {len(energy_means)} steps")
                
                try:
                    plt.figure(figsize=(12, 5))
                    
                    plt.subplot(1, 2, 1)
                    plt.plot(steps, energy_means, 'b-', linewidth=1.5, alpha=0.8, label='Energy')
                    if np.any(energy_stds > 0):
                        plt.fill_between(steps, energy_means - energy_stds, energy_means + energy_stds, 
                                       alpha=0.3, label='±1σ uncertainty')
                    plt.axhline(y=final_energy, color='r', linestyle='--', alpha=0.7, label=f'Final: {final_energy:.6f} Ha')
                    plt.xlabel('Training Step')
                    plt.ylabel('Energy (Ha)')
                    plt.title('H₂ ML-VMC Energy Convergence')
                    plt.legend()
                    plt.grid(True, alpha=0.3)
                    
                    plt.subplot(1, 2, 2)
                    initial_energy = energy_means[0]
                    energy_improvement = (energy_means - initial_energy) * 1000
                    plt.plot(steps, energy_improvement, 'g-', linewidth=1.5)
                    plt.axhline(y=0, color='k', linestyle=':', alpha=0.5)
                    plt.xlabel('Training Step')
                    plt.ylabel('Energy Change (mHa)')
                    plt.title('H₂ Energy Improvement from Initial')
                    plt.grid(True, alpha=0.3)
                    
                    plt.tight_layout()
                    
                    os.makedirs('h2_ml_runs', exist_ok=True)
                    plt.savefig('h2_ml_runs/h2_energy_convergence.png', dpi=300, bbox_inches='tight')
                    print("  ✓ Energy convergence plot saved: h2_ml_runs/h2_energy_convergence.png")
                    plt.show()
                    
                except Exception as e:
                    print(f"  ⚠ Could not create energy plots: {e}")
                
                return {'energy': final_energy, 'error': energy_std, 'training_steps': len(energy_means)}
    
    return None




h2_ml_result = analyze_h2_results()

H2 ML-VMC Results Analysis
Results file found, analyzing data...
  Available data: []
